# QwenPRM Wrapper — Score Inspection

Inspect the `QwenPRM` wrapper class (`reward_models.py`) through
the shared `score(questions, answers, batch_size=...)` interface.

Two examples: (1) the flamingo problem — single question, single
answer; (2) a batched call combining the flamingo question with
the algebra correct/wrong pair — two questions, mixed answer
counts, scored in one `prm.score()` call.

Note: the model card specifies `bfloat16`, but the V100 (sm_70)
has no bf16 support, so we load `float16`. fp16 preserves step
*rankings* but can drift absolute scores slightly.

Env: runs under `py311` (transformers 4.57). The bundled remote
code (`modeling_qwen2_rm.py`) calls a cache API removed in newer
transformers; `QwenPRM` uses `use_cache=False` to sidestep it.

## Setup

In [ ]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)
logging.disable(logging.CRITICAL)

import gc
import sys
sys.path.append("..")

import torch

from notebook_utils import gpu_mem_used_gb, print_step_scores
from core.reward_models import QwenPRM

In [2]:
# Model paths
base_dir = "/groups/chichengz/tnn/datasets"
prm_dir = f"{base_dir}/Qwen2.5-Math-PRM-7B"

## Load the PRM

In [3]:
# fp16 for V100 (sm_70); model card recommends bf16 (Ampere+),
# so absolute scores may drift slightly on Ampere GPUs.
prm = QwenPRM(prm_dir)

print(f"sep token id  : {prm.sep_token_id}")
print(f"dtype         : {next(prm.model.parameters()).dtype}")
print(f"GPU memory used: {gpu_mem_used_gb():.2f} GB")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

sep token id  : 151651
dtype         : torch.float16
GPU memory used: 29.99 GB


## Example 1 — flamingo problem (single question, single answer)

Model-card reference scores (bf16): `[1.0, 0.1904, 0.9766, 1.0]`.
On V100/fp16 expect close, not exact.

In [4]:
# Toy example from the Qwen2.5-Math-PRM-7B model card.
# Double-backslash LaTeX so "\t"/"\b" don't become control chars.
problem = (
    "Sue lives in a fun neighborhood.  One weekend, the "
    "neighbors decided to play a prank on Sue.  On Friday "
    "morning, the neighbors placed 18 pink plastic flamingos "
    "out on Sue's front yard.  On Saturday morning, the "
    "neighbors took back one third of the flamingos, painted "
    "them white, and put these newly painted white flamingos "
    "back out on Sue's front yard.  Then, on Sunday morning, "
    "they added another 18 pink plastic flamingos to the "
    "collection. At noon on Sunday, how many more pink "
    "plastic flamingos were out than white plastic flamingos?"
)

reasoning_steps = [
    "To find out how many more pink plastic flamingos were "
    "out than white plastic flamingos at noon on Sunday, we "
    "can break down the problem into steps. First, on Friday, "
    "the neighbors start with 18 pink plastic flamingos.",

    "On Saturday, they take back one third of the flamingos. "
    "Since there were 18 flamingos, (1/3 \\times 18 = 6) "
    "flamingos are taken back. So, they have (18 - 6 = 12) "
    "flamingos left in their possession. Then, they paint "
    "these 6 flamingos white and put them back out on Sue's "
    "front yard. Now, Sue has the original 12 pink flamingos "
    "plus the 6 new white ones. Thus, by the end of Saturday, "
    "Sue has (12 + 6 = 18) pink flamingos and 6 white "
    "flamingos.",

    "On Sunday, the neighbors add another 18 pink plastic "
    "flamingos to Sue's front yard. By the end of Sunday "
    "morning, Sue has (18 + 18 = 36) pink flamingos and "
    "still 6 white flamingos.",

    "To find the difference, subtract the number of white "
    "flamingos from the number of pink flamingos: "
    "(36 - 6 = 30). Therefore, at noon on Sunday, there were "
    "30 more pink plastic flamingos out than white plastic "
    "flamingos. The answer is (\\boxed{30}).",
]

In [5]:
# One question, one candidate answer whose steps are joined by
# "\n\n". score() returns [question][answer][step].
scores = prm.score([problem], [["\n\n".join(reasoning_steps)]])

print("=== Flamingo trajectory ===")
print_step_scores(reasoning_steps, scores[0][0])

=== Flamingo trajectory ===
Step 1: P(correct) = 0.9995
To find out how many more pink plastic flamingos were out th...
Step 2: P(correct) = 0.1580
On Saturday, they take back one third of the flamingos. Sinc...
Step 3: P(correct) = 0.9741
On Sunday, the neighbors add another 18 pink plastic flaming...
Step 4: P(correct) = 0.9995
To find the difference, subtract the number of white flaming...


## Example 2 — batched call (two questions, mixed answer counts)

One `prm.score()` call with two questions: the flamingo problem
(one answer) and the algebra problem (two candidate answers —
correct and wrong). The base class flattens all pairs, scores in
one batched forward pass, and reshapes back to
`[question][answer][step]`.

In [6]:
algebra_problem = "If 3x + 5 = 17, what is x?"

correct_steps = [
    "We need solve the equation 3x + 5 = 17.",
    "Subtracting 5 from both sides gives 3x = 12.",
    "Dividing both sides by 3 gives x = 4.",
    "Therefore, the answer is (\\boxed{4}).",
]

wrong_steps = [
    "Subtracting 5 from both sides gives 3x = 12.",
    "Dividing both sides by 2 gives x = 6.",
    "Therefore, the final answer is \\boxed{6}.",
]

In [7]:
# Two questions; flamingo has 1 answer, algebra has 2.
# score() flattens to 3 pairs, batches, reshapes back.
questions = [problem, algebra_problem]
answers = [
    ["\n\n".join(reasoning_steps)],
    ["\n\n".join(correct_steps), "\n\n".join(wrong_steps)],
]

batch_scores = prm.score(questions, answers, batch_size=4)

print("=== Flamingo (Q0, A0) ===")
print_step_scores(reasoning_steps, batch_scores[0][0])

print("\n=== Algebra correct (Q1, A0) ===")
print_step_scores(correct_steps, batch_scores[1][0])

print("\n=== Algebra wrong — step 2 divides by 2 (Q1, A1) ===")
print_step_scores(wrong_steps, batch_scores[1][1])

=== Flamingo (Q0, A0) ===
Step 1: P(correct) = 0.9995
To find out how many more pink plastic flamingos were out th...
Step 2: P(correct) = 0.1581
On Saturday, they take back one third of the flamingos. Sinc...
Step 3: P(correct) = 0.9741
On Sunday, the neighbors add another 18 pink plastic flaming...
Step 4: P(correct) = 0.9995
To find the difference, subtract the number of white flaming...

=== Algebra correct (Q1, A0) ===
Step 1: P(correct) = 0.9995
We need solve the equation 3x + 5 = 17.
Step 2: P(correct) = 0.9995
Subtracting 5 from both sides gives 3x = 12.
Step 3: P(correct) = 1.0000
Dividing both sides by 3 gives x = 4.
Step 4: P(correct) = 1.0000
Therefore, the answer is (\boxed{4}).

=== Algebra wrong — step 2 divides by 2 (Q1, A1) ===
Step 1: P(correct) = 0.9922
Subtracting 5 from both sides gives 3x = 12.
Step 2: P(correct) = 0.0102
Dividing both sides by 2 gives x = 6.
Step 3: P(correct) = 0.5127
Therefore, the final answer is \boxed{6}.


## Example 3 — preamble as Step 0

Qwen-Math-Instruct opens a *fresh* generation with a lead-in
sentence ("To find the angle…", "To solve this…") instead of
jumping to a labelled first step — see
`examine_llm_generation_templates_qwen_v1`. In the search tree
this preamble is the **root step**. Question: does the PRM
penalise it?

Here we prepend that preamble as **Step 0** to the algebra
*correct* trajectory from Example 2 and score the whole thing.
Two things to read off:

1. **Step 0 score** — does the PRM mark a content-free lead-in as
   low-reward (a "bad step")?
2. **Steps 1–4 vs Example 2** — does inserting the preamble
   depress the downstream steps, or do they keep their high
   scores (≈0.9995, 0.9995, 1.0, 1.0)?

Same `prm.score()` interface; the preamble is just another
`\n\n`-separated step, so it gets its own `<extra_0>` position.

In [8]:
# Algebra correct trajectory (from Example 2) with a content-free
# preamble prepended as Step 0 — the lead-in Qwen-Math emits on a
# fresh turn. Steps 1-4 are unchanged, so their scores are directly
# comparable to Example 2 (0.9995, 0.9995, 1.0, 1.0).
preamble = "To solve this problem, we will follow these steps:"
preamble_steps = [preamble, *correct_steps]

# Control: same opening but as a labelled "## Step 1:" working step
# instead of a bare preamble, to separate "no label" from "no work".
labelled_steps = [
    "## Step 1: We need solve the equation 3x + 5 = 17.",
    "## Step 2: Subtracting 5 from both sides gives 3x = 12.",
    "## Step 3: Dividing both sides by 3 gives x = 4.",
    "## Step 4: Therefore, the answer is (\\boxed{4}).",
]

ex3_scores = prm.score(
    [algebra_problem, algebra_problem],
    [["\n\n".join(preamble_steps)], ["\n\n".join(labelled_steps)]],
)

print("=== Algebra correct WITH preamble as Step 0 ===")
print_step_scores(preamble_steps, ex3_scores[0][0])

print("\n=== Algebra correct, '## Step N:'-labelled (control) ===")
print_step_scores(labelled_steps, ex3_scores[1][0])

=== Algebra correct WITH preamble as Step 0 ===
Step 1: P(correct) = 0.9902
To solve this problem, we will follow these steps:
Step 2: P(correct) = 0.9995
We need solve the equation 3x + 5 = 17.
Step 3: P(correct) = 0.9995
Subtracting 5 from both sides gives 3x = 12.
Step 4: P(correct) = 0.9995
Dividing both sides by 3 gives x = 4.
Step 5: P(correct) = 1.0000
Therefore, the answer is (\boxed{4}).

=== Algebra correct, '## Step N:'-labelled (control) ===
Step 1: P(correct) = 0.9995
## Step 1: We need solve the equation 3x + 5 = 17.
Step 2: P(correct) = 0.9990
## Step 2: Subtracting 5 from both sides gives 3x = 12.
Step 3: P(correct) = 0.9980
## Step 3: Dividing both sides by 3 gives x = 4.
Step 4: P(correct) = 1.0000
## Step 4: Therefore, the answer is (\boxed{4}).


## Example 4 — real generated preamble (verbatim model output)

Example 3 used a hand-written preamble on a clean trajectory.
Here we score the **verbatim** Qwen2.5-Math-7B-Instruct output
from `examine_llm_generation_templates_qwen_v1` (native template,
angle-between-lines question): the native/fresh preamble as
**Step 0**, then the real Step 1/2 (continuation prefix) and the
generated Step 3.

The trajectory is **incomplete** — it stops at the dot product,
has no `\boxed{}`, and the partial work doesn't reach the true
answer (90°). That's fine: we only compare the per-step score of
the preamble (Step 0) against the labelled `## Step N` steps in
the same trajectory, not final-answer correctness. This is the
realistic version of Example 3's question — does the PRM penalise
the model's *actual* root output?

In [9]:
# Verbatim Qwen2.5-Math-7B-Instruct output from
# examine_llm_generation_templates_qwen_v1 (native template).
gen_question = (
    "The set of points $(x,y,z)$ that satisfy\n\\[2x = 3y = -z\\]"
    "is a line.\n\nThe set of points $(x,y,z)$ that satisfy\n"
    "\\[6x = -y = -4z\\]is another line.\n\nFind the angle "
    "between these lines, in degrees."
)

# Step 0 = the native/fresh preamble the model emitted; Steps 1-2 =
# the continuation prefix; Step 3 = the generated continuation.
gen_steps = [
    "To find the angle between the two lines given by the "
    "equations \\(2x = 3y = -z\\) and \\(6x = -y = -4z\\), we "
    "first need to determine the direction vectors of these "
    "lines.",

    "## Step 1: Identify the direction vectors of the lines.\n"
    "For the first line the direction vector is (2, 3, -1); for "
    "the second line it is (6, -1, -4).",

    "## Step 2: Recall the formula for the angle between "
    "vectors.\ncos(theta) = (a . b) / (|a| |b|).",

    "## Step 3: Compute the dot product of the direction "
    "vectors.\na . b = 2*6 + 3*(-1) + (-1)*(-4) = 12 - 3 + 4 = 13.",
]

gen_scores = prm.score([gen_question], [["\n\n".join(gen_steps)]])

print("=== Generated trajectory, preamble as Step 0 ===")
print_step_scores(gen_steps, gen_scores[0][0])

=== Generated trajectory, preamble as Step 0 ===
Step 1: P(correct) = 0.9990
To find the angle between the two lines given by the equatio...
Step 2: P(correct) = 0.0598
## Step 1: Identify the direction vectors of the lines.
For ...
Step 3: P(correct) = 0.9609
## Step 2: Recall the formula for the angle between vectors....
Step 4: P(correct) = 0.9526
## Step 3: Compute the dot product of the direction vectors....


## Cleanup

In [10]:
del prm
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory used: {gpu_mem_used_gb():.2f} GB")

GPU memory used: 29.69 GB
